In [ ]:
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import T5Tokenizer, T5ForConditionalGeneration
import matplotlib.pyplot as plt

In [ ]:
## LOAD MODELS

model_name = "google/flan-t5-large"
tokeniser = T5Tokenizer.from_pretrained(model_name)

model_loc1 = "path/to/ts1"
model_loc2 = "path/to/ts2.1"
model_loc3 = "path/to/ts2.2"
model_loc4 = "path/to/ts2.3"
model_loc5 = "path/to/ts2.4" # or alternatively ts2.4.1, ts2.4.2


model0 = T5ForConditionalGeneration.from_pretrained(model_name,
                                                   torch_dtype=torch.bfloat16,
                                                   device_map='cuda')
model1 = T5ForConditionalGeneration.from_pretrained(model_loc1,
                                                   torch_dtype=torch.bfloat16,
                                                   device_map='cuda')
model21 = T5ForConditionalGeneration.from_pretrained(model_loc2,
                                                   torch_dtype=torch.bfloat16,
                                                   device_map='cuda')
model22 = T5ForConditionalGeneration.from_pretrained(model_loc3,
                                                   torch_dtype=torch.bfloat16,
                                                   device_map='cuda')
model23 = T5ForConditionalGeneration.from_pretrained(model_loc4,
                                                   torch_dtype=torch.bfloat16,
                                                   device_map='cuda')
model24 = T5ForConditionalGeneration.from_pretrained(model_loc5,
                                                   torch_dtype=torch.bfloat16,
                                                   device_map='cuda')

In [ ]:
df = pd.read_csv('path/to/V_standard.csv') # load evaluation set

# obtain sample, and the sentence, cause, and effect
row = df.iloc[9149]
input_text = row['sentence']
cause_text = row['cause']
effect_text = row['effect']

print("Input:", input_text)
print("Label:", row['label'])
print("Cause:", cause_text)
print("Effect:", effect_text)


P_eval = [
    "Taking the statement '{input}' into account, is '{cause}' the event that made '{effect}' occur?",
    "If '{input}' holds true, can we definitively state that '{effect}' was brought about by '{cause}'?",
    "Does '{cause}' directly explain why '{effect}' took place, based on the information: '{input}'?",
    "Judging from '{input}', is it correct to link the occurrence of '{effect}' back to '{cause}'?",
    "If we consider '{input}', does it logically suggest that '{cause}' is behind the event '{effect}'?",
    "Based on your understanding of '{input}', would you say that '{cause}' instigated '{effect}'?",
    "Reviewing '{input}', is it valid to claim '{cause}' was the catalyst for '{effect}'?",
    "Does the provided scenario '{input}' imply that the event '{effect}' was the outcome caused by '{cause}'?"
]


paraphrased_inputs = [fmt.format(input=input_text, cause=cause_text, effect=effect_text) for fmt in P_eval] # apply formats to sample

models = [model0, model1, model21, model22, model23, model24]

In [ ]:
model_names = ['ts0', 'ts1', 'ts2.1', 'ts2.2','ts2.3', 'ts2.4']

for num, model in enumerate(models):
    inputs = tokeniser(paraphrased_inputs, return_tensors='pt', truncation=True, padding=True, max_length=512).to("cuda") # tokenise inputs
    decoder_input = torch.full((8, 1), model.config.decoder_start_token_id, device="cuda")

    with torch.no_grad():
        # get the model outputs, and the representations
        answer_reps = model.generate(**inputs, max_length=2, output_scores=True, return_dict_in_generate=True, output_hidden_states=True)
        decoder_hs = answer_reps.decoder_hidden_states[0]
        answers = model.generate(**inputs, max_length=2)

    trajectories = [[] for _ in range(len(P_eval))] # list of trajectories for each paraphrase format
    for layer_num, layer in enumerate(decoder_hs[1:]): # iterate over representation at each decoder layer
        representation = layer[:, -1, :].to(torch.float32) # get representation for layer
        yes_logit = torch.matmul(representation, model.lm_head.weight[4273].to(torch.float32)) # obtain values for 'yes' and 'no' logits
        no_logit  = torch.matmul(representation, model.lm_head.weight[150].to(torch.float32))

        # find L2 normalised difference between logits
        logits_2d = torch.stack([yes_logit, no_logit])
        normed = F.normalize(logits_2d.unsqueeze(0), p=2, dim=1).squeeze(0)
        scalar = (normed[0] - normed[1] + 1) / 2

        for i in range(len(P_eval)):
            trajectories[i].append(scalar[i].item()) # append difference value for each format

    num_layers = len(decoder_hs) - 1
    x_range = list(range(1, num_layers+1))

    # configure plot, and plot trajectories
    plt.figure(figsize=(8, 6))
    plt.ylim(-0.3, 1.3)
    plt.xlim(0, num_layers+1)
    plt.xticks([1, 5, 10, 15, 20, 24])
    plt.axhspan(0.5, 1.3, facecolor='orange', alpha=0.2)
    plt.axhspan(-0.3, 0.5, facecolor='blue', alpha=0.2)
    plt.text(0.5, 0.75, "YES", fontsize=40, color='orange', alpha=0.5,
             ha='center', va='center', transform=plt.gca().transAxes)
    plt.text(0.5, 0.25, "NO", fontsize=40, color='blue', alpha=0.5,
             ha='center', va='center', transform=plt.gca().transAxes)
    for idx, path in enumerate(trajectories):
        plt.plot(x_range, path, marker='o', label=f"P_eval({idx+1})")

    # obtain consistency metrics for plot
    mean_vec = torch.mean(representation, dim=0)
    distances = torch.norm(representation - mean_vec, dim=1).mean().item()
    sim = F.cosine_similarity(representation, mean_vec, dim=1).mean().item()
    abs = sum([1 for i in answers if i[1] == 4273 or i[1] == 2163]) / 0.08 # divide by 8, then multiply by 100 to get percentage
    if abs < 50:
        abs = 100 - abs

    # plot the representational consistency metrics
    plt.text(0.2, 1.285, f"Rep. Consis. (MSE): {distances:.3f}",
            fontsize=12, verticalalignment='top',
            bbox=dict(facecolor='white', alpha=0.6))
    plt.text(0.2, 1.185, f"Rep. Consis. (CS): {sim:.3f}",
            fontsize=12, verticalalignment='top',
            bbox=dict(facecolor='white', alpha=0.6))
    plt.text(0.2, 1.085, f"Ans. Consis.: {abs:.1f}%",
            fontsize=12, verticalalignment='top',
            bbox=dict(facecolor='white', alpha=0.6))

    plt.axhline(0.5, color='black', linestyle='--')

    plt.savefig(f'{model_names[num]}.png')
    plt.show()
    plt.close()
